In [6]:
from fastapi import APIRouter, Query, HTTPException
from typing import Optional, List, Dict, Any, Literal
from datetime import datetime, timedelta
from collections import defaultdict
import pandas as pd
import json
import re
import sys

sys.path.insert(0, r"D:/A0_Project")  # 讓它優先被找到
from PI_SYSTEM.models.sql_db_connect import MySQLConnet

db = MySQLConnet('piaoi_ol_defect_map')
cim_db = MySQLConnet('cim_piaoi')
old_db = MySQLConnet('l6a01_project')

#db = MySQLConnet('piaoi_inspection_density')
#from PI_SYSTEM.models.density.cim_density_job import Config as DensityJobConfig
#from PI_SYSTEM.models.density.API_Config import API_Config

In [5]:
old_rdf = old_db.get_runs_delta_days('aoi200_rawdata_202604', 3, 'scantime')
print(len(old_rdf))
old_sdf = old_db.get_runs_delta_days('aoi_summary_aoi200', 3, 'scantime')
print(len(old_sdf))

175497
669


In [3]:
cim_rdf = cim_db.get_runs_delta_days('cim_defect_202604_aoi200_capic100', 3, 'test_time')
print(len(cim_rdf))
cim_sdf = cim_db.get_runs_delta_days('cim_pi_glass_202604', 3, 'test_time')
print(len(cim_sdf))


22883
1005


In [ ]:
#print(old_rdf.iloc[0].to_dict())
print(old_sdf.iloc[0].to_dict())
print(cim_rdf.iloc[0].to_dict())
print(cim_sdf.iloc[0].to_dict())

{'scantime': Timestamp('2026-04-08 15:15:37'), 'line_id': 'CAAOI202', 'model': 'M215HAN01-T-D-API', 'glass_type': 'M215HAN01-T-D-API', 'recipe_id': '2231', 'glass_id': 'LV6A3WH3A', 'pic_name': 'CaptureImage/RV1_551911_1158498_96', 'x': '551911', 'y': '1158498', 'predict_code': '1', 'judge_code': '1', 'mark': '', 'hour': '', 'dayhour': '', 'day': Timestamp('2026-04-08 15:15:37'), 'year': '', 'month': '', 'season': '', 'week': '', 'yearmonth': '', 'defect_count': '375', 'defect_size': '76', 'open_status': '', 'ai_code_1': '', 'ai_code_2': '', 'ai_code_3': '', 'ai_code_4': '', 'ai_code_5': '', 'gray_name': '', 'ip_num': '', 'first_code': '', 'chip_name': 'D', 'defect_seq': '99', 'image_path': '/CAAOI202/CA001103/LV6A3WH3A/PCS1/20260408151537/', 'over_defect_count': '1', 'large_defect_count': '30', 'middle_defect_count': '297', 'small_defect_count': '47', 'indexid': '0188b283a26c40448e65c72d621e50d4'}
{'run_day': datetime.date(2026, 4, 8), 'scantime': Timestamp('2026-04-08 15:15:37'), 'lin

In [ ]:
db.list_tables()


2026-04-09 15:23:08,703 - INFO - [list_tables] 成功取得資料表名稱，共 1 張表。


['aoi200_202604_api_summary_table']

In [17]:
rows = cim_db.get_runs_delta_days('cim_defect_202604_aoi200_capic100', 2, 'test_time')
rows = rows[rows['sheet_id_chip_id'] =='AF6A3U03A']
rows

,sheet_id_chip_id,chip_id,test_time,defect_size,pox_x1,pox_y1,image_file_path,image_file_name,img_file_url_path,retype_def_code,adc_def_code,pi_time,pi_hour
280,AF6A3U03A,0JE,2026-04-08 14:46:17,S,1279372,417502,Image/CA031401/AF6A3U03A/,RV1_1279372_417502_5.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,PI_Spot_NP,2026-04-08 10:21:11,2026-04-08 09:00:00
281,AF6A3U03A,0EF,2026-04-08 14:46:17,S,714082,59800,Image/CA031401/AF6A3U03A/,RV1_714082_59800_4.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
282,AF6A3U03A,0FC,2026-04-08 14:46:17,L,852995,929974,Image/CA031401/AF6A3U03A/,RV1_852995_929974_0.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
283,AF6A3U03A,0FC,2026-04-08 14:46:17,M,852823,930090,Image/CA031401/AF6A3U03A/,RV1_852823_930090_1.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
284,AF6A3U03A,0LB,2026-04-08 14:46:17,S,1585282,1208619,Image/CA031401/AF6A3U03A/,RV1_1585282_1208619_3.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1446/,nan,Not_Found,2026-04-08 10:21:11,2026-04-08 09:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
389,AF6A3U03A,0KD,2026-04-08 14:43:50,S,1460476,698819,Image/CA031401/AF6A3U03A/,RV1_1460476_698819_94.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00
390,AF6A3U03A,0KD,2026-04-08 14:43:50,S,1445916,533188,Image/CA031401/AF6A3U03A/,RV1_1445916_533188_95.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00
391,AF6A3U03A,0KF,2026-04-08 14:43:50,S,1391848,188826,Image/CA031401/AF6A3U03A/,RV1_1391848_188826_100.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00
392,AF6A3U03A,0CC,2026-04-08 14:43:50,S,341023,803002,Image/CA031401/AF6A3U03A/,RV1_341023_803002_4.jpg,PIT/2604/08/CAAOI202/AF6A3U03A/1443/,nan,SSIU_Polymer,2026-04-08 10:21:11,2026-04-08 09:00:00


In [18]:
rows['test_time'].unique()

<DatetimeArray>
['2026-04-08 14:46:17', '2026-04-08 14:43:50']
Length: 2, dtype: datetime64[ns]

In [16]:
df = db.get_table('aoi200_202604_api_summary_table')
print(len(df))
df.iloc[15].to_dict()

38


{'id': 16,
 'sheet_id_chip_id': 'AF6A3U03A',
 'test_time': Timestamp('2026-04-08 14:46:17'),
 'recipe_id': '0250',
 'line_id': 'CAPIC100',
 'aoi': 'aoi200',
 'defect_count': 6,
 'small_defect_count': 3,
 'middle_defect_count': 2,
 'large_defect_count': 1,
 'over_defect_count': 0,
 'pi_time': Timestamp('2026-04-08 10:21:11'),
 'pi_type': 'API',
 'run_day': Timestamp('2026-04-08 00:00:00'),
 'update_time': Timestamp('2026-04-09 15:22:29')}

In [19]:
df[df['sheet_id_chip_id'] =='AF6A3U03A']

,id,sheet_id_chip_id,test_time,recipe_id,line_id,aoi,defect_count,small_defect_count,middle_defect_count,large_defect_count,over_defect_count,pi_time,pi_type,run_day,update_time
14,15,AF6A3U03A,2026-04-08 14:43:50,2250,CAPIC100,aoi200,108,76,31,1,0,2026-04-08 10:21:11,API,2026-04-08,2026-04-09 15:22:29
15,16,AF6A3U03A,2026-04-08 14:46:17,0250,CAPIC100,aoi200,6,3,2,1,0,2026-04-08 10:21:11,API,2026-04-08,2026-04-09 15:22:29
